# Instalamos dependencias

In [1]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install plotly jinja2 weasyprint kaleido pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install nbformat --upgrade

Note: you may need to restart the kernel to use updated packages.


# Cargamos el dataset
### Seleccionamos la universidad de la que generaremos el informe

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/danielgomez/Desktop/TFG/data/clean-data/La_Laguna/Tabla.csv", sep=';')

# 2. Limpieza de seguridad (por si quedaron ??? o ..)
def limpiar_valor(v):
    if v in ["??", "???", "..", "None"] or pd.isna(v):
        return np.nan
    return float(str(v).replace(',', '.'))

df['Valor'] = df['Valor'].apply(limpiar_valor)

# Generamos grafico

In [6]:
# Colores institucionales
carrera = "Informática"
colores_genero = {"Hombres": "#1f77b4", "Mujeres": "#e377c2", "Ambos Sexos": "#7f7f7f"}
df_filtrado = df[df['Carrera'] == carrera]

## Faceted Bar Chart

In [ ]:
import plotly.express as px

fig = px.bar(
        df_filtrado, 
        x="Anio", y="Valor", color="Genero",
        facet_col="Tasa", # Divide en 3 columnas por cada tasa
        barmode="group",
        title=f"Panel de Control Académico: {carrera}",
        labels={"Valor": "Porcentaje (%)", "Anio": "Curso Académico"},
        color_discrete_map=colores_genero,
        text_auto='.1f'
    )
fig.update_xaxes(autorange="reversed")
fig.write_image(f"../output/graph/G1_Resumen_{carrera}.png")
fig.show()

## Evolución de la Brecha Neta (Line Chart)

In [ ]:
import plotly.express as px

df_pivot = df_filtrado.pivot_table(index=["Anio", "Tasa"], columns="Genero", values="Valor").reset_index()
df_pivot['Brecha'] = df_pivot['Mujeres'] - df_pivot['Hombres']

fig = px.line(
    df_pivot, x="Anio", y="Brecha", color="Tasa",
    title=f"Evolución de la Brecha de Género (M - H): {carrera}",
    markers=True
)
fig.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Igualdad")
fig.update_xaxes(autorange="reversed")
fig.write_image(f"../output/graph/G2_Brecha_{carrera}.png")
fig.show()

## Velocímetro de Rendimiento

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Obtenemos el último año y la tasa de rendimiento
ultimo_anio = df_filtrado['Anio'].max()
valor_actual = df_filtrado[(df_filtrado['Anio'] == ultimo_anio) & 
                            (df_filtrado['Tasa'] == "Rendimiento") & 
                            (df_filtrado['Genero'] == "Ambos Sexos")]['Valor'].values[0]

fig = go.Figure(go.Indicator(
    mode = "gauge+number",
    value = valor_actual,
    title = {'text': f"Rendimiento Actual ({ultimo_anio})"},
    gauge = {'axis': {'range': [0, 100]},
                'bar': {'color': "#1f77b4"},
                'steps': [
                    {'range': [0, 50], 'color': "#ff9999"},
                    {'range': [50, 80], 'color': "#ffff99"},
                    {'range': [80, 100], 'color': "#99ff99"}]}
))
fig.write_image(f"../output/graph/G3_Velocimetro_{carrera}.png")
fig.show()

## Comparativa contra la Media (Bullet Chart)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

ultimo_anio = df['Anio'].max()
# Valor de la carrera
val_carrera = df[(df['Carrera'] == carrera) & (df['Anio'] == ultimo_anio) & (df['Tasa'] == "Rendimiento") & (df['Genero'] == "Ambos Sexos")]['Valor'].values[0]
# Media global
val_media = df[(df['Carrera'] == "Todos los ámbitos") & (df['Anio'] == ultimo_anio) & (df['Tasa'] == "Rendimiento") & (df['Genero'] == "Ambos Sexos")]['Valor'].values[0]

fig = go.Figure(go.Indicator(
    mode = "number+gauge+delta",
    value = val_carrera,
    delta = {'reference': val_media},
    domain = {'x': [0, 1], 'y': [0, 1]},
    title = {'text': f"Rendimiento {carrera} vs Media Univ."},
    gauge = {
        'shape': "bullet",
        'axis': {'range': [None, 100]},
        'threshold': {
            'line': {'color': "red", 'width': 2},
            'thickness': 0.75,
            'value': val_media}, # La línea roja es la media de la universidad
        'bar': {'color': "#1f77b4"}
    }
))
fig.write_image(f"../output/graph/G4_CompMedia_{carrera}.png")
fig.show()

## Tabla de Resumen Estadístico

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Resumen de los últimos 3 años
df_tabla = df_filtrado[df_filtrado['Genero'] != "Ambos Sexos"].tail(9) 

fig = go.Figure(data=[go.Table(
    header=dict(values=['Año', 'Tasa', 'Género', 'Valor (%)'],
                fill_color='paleturquoise', align='left'),
    cells=dict(values=[df_tabla.Anio, df_tabla.Tasa, df_tabla.Genero, df_tabla.Valor],
                fill_color='lavender', align='left'))
])
fig.show()

## Lineas de rendimiento (Hombres, Mujeres, ambos sexos)

In [9]:
import plotly.express as px

df_rendimiento = df[(df['Tasa'] == "Rendimiento") & (df['Valor'] > 0) & (df['Carrera'] == carrera)]

fig = px.line(
    df_rendimiento, x="Anio", y="Valor", color="Genero",
    title=f"Evolución del Rendimiento: {carrera}",
    markers=True,
    labels={"Valor": "Porcentaje (%)", "Anio": "Curso Académico"},
    color_discrete_map=colores_genero,
)

fig.show()

## Comparación con la media de la universidad

In [7]:
import plotly.graph_objects as go

ultimo_anio = df['Anio'].max()
# print(ultimo_anio)
# Valor de la carrera
val_carrera = df[(df['Carrera'] == carrera) & (df['Anio'] == ultimo_anio) & (df['Tasa'] == "Rendimiento") & (df['Genero'] == "Ambos Sexos")]['Valor'].values[0]
# Media global
val_media = df[(df['Carrera'] == "Todos los ámbitos") & (df['Anio'] == ultimo_anio) & (df['Tasa'] == "Rendimiento") & (df['Genero'] == "Ambos Sexos")]['Valor'].values[0]

fig = go.Figure()

# Barra de la carrera
fig.add_trace(go.Bar(
    x=[carrera],
    y=[val_carrera],
    name=carrera,
    marker_color='#1f77b4',
    text=[f"{val_carrera}%"],
    textposition='auto',
))

# Línea horizontal para la media
fig.add_shape(
    type="line",
    x0=-0.5, x1=0.5, y0=val_media, y1=val_media,
    line=dict(color="Red", width=3, dash="dashdot"),
)

# Anotación para la media
fig.add_annotation(
    x=0.5, y=val_media,
    text=f"Media Univ: {val_media}%",
    showarrow=False,
    yshift=10,
    font=dict(color="red", size=12)
)

fig.update_layout(
    title=f"Rendimiento en {carrera} vs Media Universidad",
    yaxis=dict(title="Porcentaje", range=[0, 100]),
    template="plotly_white",
    height=500,
    width=600
)

fig.show()

## Tabla datos del ultimo año

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

ultimo_anio = df['Anio'].max()

# Resumen de los últimos 3 años
df_tabla = df_filtrado[(df_filtrado['Genero'] != "Ambos Sexos") & (df_filtrado['Carrera'] == carrera) & (df_filtrado['Anio'] >= ultimo_anio)]

fig = go.Figure(data=[go.Table(
    header=dict(values=['Tasa', 'Género', 'Valor (%)'],
                fill_color='paleturquoise', align='left'),
    cells=dict(values=[df_tabla.Tasa, df_tabla.Genero, df_tabla.Valor],
                fill_color='lavender', align='left'))
])
fig.show()

## Brecha Hombres vs Mujeres

In [19]:
import plotly.graph_objects as go

# 1. Preparar los datos
# Filtramos para quedarnos solo con Hombres y Mujeres y la tasa específica
df_gap = df[(df['Genero'].isin(['Mujeres', 'Hombres'])) & 
            (df['Tasa'] == "Rendimiento") & 
            (df['Carrera'] == "Informática")].copy()

# Pivotamos para tener Hombres y Mujeres como columnas
df_pivot = df_gap.pivot(index='Anio', columns='Genero', values='Valor').reset_index()

# Ordenar por año para que la evolución sea lógica
df_pivot = df_pivot.sort_values('Anio')

fig = go.Figure()

# 2. Añadir las líneas de la "mancuerna" (Conectores)
for i, row in df_pivot.iterrows():
    fig.add_shape(
        type="line",
        x0=row['Hombres'], x1=row['Mujeres'],
        y0=row['Anio'], y1=row['Anio'],
        line=dict(color="#dcdcdc", width=3)
    )

# 3. Añadir puntos de Hombres
fig.add_trace(go.Scatter(
    x=df_pivot['Hombres'], 
    y=df_pivot['Anio'],
    mode='markers',
    name='Hombres',
    marker=dict(color='#1f77b4', size=12, symbol='circle'),
    hovertemplate="Hombres: %{x}%<extra></extra>"
))

# 4. Añadir puntos de Mujeres
fig.add_trace(go.Scatter(
    x=df_pivot['Mujeres'], 
    y=df_pivot['Anio'],
    mode='markers',
    name='Mujeres',
    marker=dict(color='#e377c2', size=12, symbol='circle'),
    hovertemplate="Mujeres: %{x}%<extra></extra>"
))

# 5. Añadir el valor de la brecha como texto a la derecha de cada mancuerna
for i, row in df_pivot.iterrows():
    brecha = row['Mujeres'] - row['Hombres']
    # Determinamos qué punto está más a la derecha para colocar el texto
    pos_x = max(row['Mujeres'], row['Hombres']) + 1 
    
    fig.add_annotation(
        x=pos_x, y=row['Anio'],
        text=f"Δ {brecha:+.2f}%",
        showarrow=False,
        xanchor="left",
        font=dict(size=10, color="gray")
    )

# 6. Estética del gráfico
fig.update_layout(
    title=dict(
        text=f"<b>Brecha de Género en Rendimiento</b><br>Carrera: Informática",
        font=dict(size=18)
    ),
    xaxis=dict(
        title="Tasa de Rendimiento (%)",
        gridcolor='#f0f0f0',
        range=[df_gap['Valor'].min() - 5, df_gap['Valor'].max() + 8] # Espacio para el texto
    ),
    yaxis=dict(title="Curso Académico", gridcolor='#f0f0f0'),
    margin=dict(l=100, r=80, t=80, b=50),
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=500
)

fig.show()

# Escribir informe

In [ ]:
pip install fpdf2

In [ ]:
import pandas as pd
from jinja2 import Template
from datetime import datetime
import os
from pathlib import Path

# 1. Configuración de datos
carrera_actual = "Informática"
fecha_hoy = datetime.now().strftime("%d/%m/%Y")

def ruta_a_uri(ruta_relativa):
    # Convierte ruta relativa a absoluta y luego a formato URI (file:///...)
    return Path(ruta_relativa).resolve().as_uri()

datos_informe = {
    "carrera": carrera_actual,
    "fecha": fecha_hoy,
    "tasa_tipo": "Rendimiento / Éxito / Evaluación",
    "tendencia_g1": "estable con ligero crecimiento en el último bienio",
    "path_g1": ruta_a_uri(f"../output/graph/G1_Resumen_{carrera_actual}.png"),
    "path_g2": ruta_a_uri(f"../output/graph/G2_Brecha_{carrera_actual}.png"),
    "path_g3": ruta_a_uri(f"../output/graph/G3_Velocimetro_{carrera_actual}.png"),
    "path_g4": ruta_a_uri(f"../output/graph/G4_CompMedia_{carrera_actual}.png"),
}

# 3. Leer la plantilla HTML
with open("../templates/informe_prueba.html", "r", encoding="utf-8") as f:
    plantilla_html = f.read()

# 4. Renderizar (inyectar datos)
template = Template(plantilla_html)
html_final = template.render(datos_informe)

# 5. Guardar el HTML resultante (para revisar o convertir)
with open(f"../output/informe_html/informe_{carrera_actual}.html", "w", encoding="utf-8") as f:
    f.write(html_final)

print(f"HTML generado para {carrera_actual}. Ahora puedes convertirlo a PDF.")

In [ ]:
pip install weasyprint

In [ ]:
from weasyprint import HTML
HTML(string=html_final).write_pdf(f"../output/informe_pdf/Informe_{carrera_actual}.pdf")